In [54]:
!pip install prophet
!pip install xgboost
!pip install pmdarima
!pip install fastapi uvicorn pyngrok nest-asyncio

In [55]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt

# Metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# Scaling
from sklearn.preprocessing import MinMaxScaler

# SARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Prophet
from prophet import Prophet

# XGBoost
from xgboost import XGBRegressor

# LSTM
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense,
    LSTM,
    Dropout
)

# Save models
import joblib

In [56]:
from google.colab import files

uploaded = files.upload()

Saving Forecasting Case- Study.xlsx to Forecasting Case- Study (2).xlsx


In [57]:
df = pd.read_excel(
    '/content/Forecasting Case- Study.xlsx'
)

In [58]:
df.head()

,State,Date,Total,Category
0,Alabama,2019-01-12 00:00:00,109574036.0,Beverages
1,Arizona,2019-01-12 00:00:00,109101594.6,Beverages
2,Arkansas,2019-01-12 00:00:00,58049432.2,Beverages
3,California,2019-01-12 00:00:00,444766890.6,Beverages
4,Colorado,2019-01-12 00:00:00,89816716.3,Beverages


In [59]:
print(df.shape)

print(df.info())

print(df.isnull().sum())

(8084, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8084 entries, 0 to 8083
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   State     8084 non-null   object 
 1   Date      8084 non-null   object 
 2   Total     8084 non-null   float64
 3   Category  8084 non-null   object 
dtypes: float64(1), object(3)
memory usage: 252.8+ KB
None
State       0
Date        0
Total       0
Category    0
dtype: int64


In [60]:
df['Date'] = pd.to_datetime(
    df['Date']
)

In [61]:
df = df.sort_values(
    ['State', 'Date']
)

In [62]:
df['Total'] = (
    df.groupby('State')['Total']
    .transform(
        lambda x: x.interpolate(
            method='linear'
        )
    )
)

df['Total'] = (
    df.groupby('State')['Total']
    .transform(
        lambda x: x.bfill().ffill()
    )
)

In [63]:
all_states = []

for state in df['State'].unique():

    temp = df[
        df['State'] == state
    ].copy()

    # Preserve category
    category_value = (
        temp['Category']
        .dropna()
        .iloc[0]
    )

    # Create complete weekly dates
    full_dates = pd.date_range(
        start=temp['Date'].min(),
        end=temp['Date'].max(),
        freq='W-SUN'
    )

    # Set index
    temp = temp.set_index('Date')

    # Reindex
    temp = temp.reindex(full_dates)

    # Restore index
    temp.index.name = 'Date'

    temp = temp.reset_index()

    # Restore columns
    temp['State'] = state

    temp['Category'] = category_value

    # Interpolate missing totals
    temp['Total'] = (
        temp['Total']
        .interpolate(method='linear')
    )

    temp['Total'] = (
        temp['Total']
        .bfill()
        .ffill()
    )

    all_states.append(temp)

# Combine all states
df = pd.concat(
    all_states,
    ignore_index=True
)

In [64]:
df.head(20)

,Date,State,Total,Category
0,2019-01-13,Alabama,123782285.8,Beverages
1,2019-01-20,Alabama,123782285.8,Beverages
2,2019-01-27,Alabama,123782285.8,Beverages
3,2019-02-03,Alabama,123782285.8,Beverages
4,2019-02-10,Alabama,123782285.8,Beverages
5,2019-02-17,Alabama,123782285.8,Beverages
6,2019-02-24,Alabama,123782285.8,Beverages
7,2019-03-03,Alabama,123782285.8,Beverages
8,2019-03-10,Alabama,123782285.8,Beverages
9,2019-03-17,Alabama,123782285.8,Beverages


In [65]:
df.isnull().sum()

,0
Date,0
State,0
Total,0
Category,0


In [66]:
def create_features(data):

    data = data.copy()

    # Lag Features
    data['lag_1'] = (
        data['Total'].shift(1)
    )

    data['lag_7'] = (
        data['Total'].shift(7)
    )

    data['lag_30'] = (
        data['Total'].shift(30)
    )

    # Rolling Mean
    data['rolling_mean_7'] = (
        data['Total']
        .shift(1)
        .rolling(7)
        .mean()
    )

    # Rolling Std
    data['rolling_std_7'] = (
        data['Total']
        .shift(1)
        .rolling(7)
        .std()
    )

    # Time Features
    data['month'] = (
        data['Date'].dt.month
    )

    data['week'] = (
        data['Date']
        .dt
        .isocalendar()
        .week
        .astype(int)
    )

    data['year'] = (
        data['Date'].dt.year
    )

    data['day_of_week'] = (
        data['Date'].dt.dayofweek
    )

    data['quarter'] = (
        data['Date'].dt.quarter
    )

    return data

In [67]:
feature_df = []

for state in df['State'].unique():

    temp = df[
        df['State'] == state
    ].copy()

    temp = create_features(temp)

    feature_df.append(temp)

df = pd.concat(
    feature_df,
    ignore_index=True
)

In [68]:
from pandas.tseries.holiday import (
    USFederalHolidayCalendar
)

cal = USFederalHolidayCalendar()

holidays = cal.holidays(
    start=df['Date'].min(),
    end=df['Date'].max()
)

df['holiday_flag'] = (
    df['Date']
    .isin(holidays)
    .astype(int)
)

In [69]:
df = df.dropna()

In [70]:
df.head()

,Date,State,Total,Category,lag_1,lag_7,lag_30,rolling_mean_7,rolling_std_7,month,week,year,day_of_week,quarter,holiday_flag
30,2019-08-11,Alabama,123782285.8,Beverages,123782285.8,123782285.8,123782285.8,123782285.8,0.0,8,32,2019,6,3,0
31,2019-08-18,Alabama,123782285.8,Beverages,123782285.8,123782285.8,123782285.8,123782285.8,0.0,8,33,2019,6,3,0
32,2019-08-25,Alabama,123782285.8,Beverages,123782285.8,123782285.8,123782285.8,123782285.8,0.0,8,34,2019,6,3,0
33,2019-09-01,Alabama,123782285.8,Beverages,123782285.8,123782285.8,123782285.8,123782285.8,0.0,9,35,2019,6,3,0
34,2019-09-08,Alabama,123782285.8,Beverages,123782285.8,123782285.8,123782285.8,123782285.8,0.0,9,36,2019,6,3,0


In [71]:
df.isnull().sum()

,0
Date,0
State,0
Total,0
Category,0
lag_1,0
lag_7,0
lag_30,0
rolling_mean_7,0
rolling_std_7,0
month,0


In [72]:
def evaluate_model(
    y_true,
    y_pred
):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    return mae, rmse

In [73]:
sarima_results = []

sarima_models = {}

In [74]:
for state in df['State'].unique():

    print(f"Training SARIMA for {state}")

    # Filter one state
    temp = df[
        df['State'] == state
    ].copy()

    # Keep only required columns
    temp = temp[
        ['Date', 'Total']
    ]

    # Set datetime index
    temp = temp.set_index('Date')

    # IMPORTANT:
    # weekly frequency
    temp = temp.asfreq('W-SUN')

    # Train-test split
    train_size = int(len(temp) * 0.8)

    train = temp.iloc[:train_size]

    test = temp.iloc[train_size:]

    try:

        # Build SARIMA model
        model = SARIMAX(
            train['Total'],

            order=(1,1,1),

            seasonal_order=(1,1,1,12),

            enforce_stationarity=False,

            enforce_invertibility=False
        )

        # Train
        fitted_model = model.fit(
            disp=False
        )

        # Predict
        preds = fitted_model.forecast(
            steps=len(test)
        )

        # Evaluate
        mae, rmse = evaluate_model(
            test['Total'],
            preds
        )

        # Store results
        sarima_results.append({

            'State': state,

            'Model': 'SARIMA',

            'MAE': mae,

            'RMSE': rmse

        })

        # Save trained model
        sarima_models[state] = fitted_model

        print(f"Completed {state}")

    except Exception as e:

        print(
            f"Error in {state}: {e}"
        )

Training SARIMA for Alabama
Completed Alabama
Training SARIMA for Arizona
Completed Arizona
Training SARIMA for Arkansas
Completed Arkansas
Training SARIMA for California
Completed California
Training SARIMA for Colorado
Completed Colorado
Training SARIMA for Connecticut
Completed Connecticut
Training SARIMA for Florida
Completed Florida
Training SARIMA for Georgia
Completed Georgia
Training SARIMA for Illinois
Completed Illinois
Training SARIMA for Indiana
Completed Indiana
Training SARIMA for Iowa
Completed Iowa
Training SARIMA for Kansas
Completed Kansas
Training SARIMA for Kentucky
Completed Kentucky
Training SARIMA for Louisiana
Completed Louisiana
Training SARIMA for Maine
Completed Maine
Training SARIMA for Maryland
Completed Maryland
Training SARIMA for Massachusetts
Completed Massachusetts
Training SARIMA for Michigan
Completed Michigan
Training SARIMA for Minnesota
Completed Minnesota
Training SARIMA for Mississippi
Completed Mississippi
Training SARIMA for Missouri
Completed

In [75]:
sarima_results[:5]

[{'State': 'Alabama',
  'Model': 'SARIMA',
  'MAE': 33451336.421217274,
  'RMSE': np.float64(36972180.694626085)},
 {'State': 'Arizona',
  'Model': 'SARIMA',
  'MAE': 32958449.275640022,
  'RMSE': np.float64(35995953.49002406)},
 {'State': 'Arkansas',
  'Model': 'SARIMA',
  'MAE': 14159675.422448557,
  'RMSE': np.float64(15728782.70103411)},
 {'State': 'California',
  'Model': 'SARIMA',
  'MAE': 144506797.79020053,
  'RMSE': np.float64(164294537.57473114)},
 {'State': 'Colorado',
  'Model': 'SARIMA',
  'MAE': 19277946.046026748,
  'RMSE': np.float64(21851728.404658392)}]

In [76]:
sarima_results_df = pd.DataFrame(
    sarima_results
)

sarima_results_df.head()

,State,Model,MAE,RMSE
0,Alabama,SARIMA,3.345134e+07,3.697218e+07
1,Arizona,SARIMA,3.295845e+07,3.599595e+07
2,Arkansas,SARIMA,1.415968e+07,1.572878e+07
3,California,SARIMA,1.445068e+08,1.642945e+08
4,Colorado,SARIMA,1.927795e+07,2.185173e+07


In [77]:
prophet_results = []

prophet_models = {}

In [78]:
for state in df['State'].unique():

    print(f"Training Prophet for {state}")

    # Filter one state
    temp = df[
        df['State'] == state
    ].copy()

    # Prophet requires:
    # ds -> date
    # y  -> target
    prophet_df = temp[
        ['Date', 'Total']
    ].copy()

    prophet_df.columns = [
        'ds',
        'y'
    ]

    # Train-test split
    train_size = int(
        len(prophet_df) * 0.8
    )

    train = prophet_df.iloc[:train_size]

    test = prophet_df.iloc[train_size:]

    try:

        # Create Prophet model
        model = Prophet(

            yearly_seasonality=True,

            weekly_seasonality=True,

            daily_seasonality=False

        )

        # Train model
        model.fit(train)

        # Create future dataframe
        future = model.make_future_dataframe(

            periods=len(test),

            freq='W'

        )

        # Predict
        forecast = model.predict(
            future
        )

        preds = forecast[
            'yhat'
        ].tail(len(test)).values

        # Evaluate
        mae, rmse = evaluate_model(

            test['y'],

            preds

        )

        # Store results
        prophet_results.append({

            'State': state,

            'Model': 'Prophet',

            'MAE': mae,

            'RMSE': rmse

        })

        # Save trained model
        prophet_models[state] = model

        print(f"Completed {state}")

    except Exception as e:

        print(
            f"Error in {state}: {e}"
        )

Training Prophet for Alabama
Completed Alabama
Training Prophet for Arizona
Completed Arizona
Training Prophet for Arkansas
Completed Arkansas
Training Prophet for California
Completed California
Training Prophet for Colorado
Completed Colorado
Training Prophet for Connecticut
Completed Connecticut
Training Prophet for Florida
Completed Florida
Training Prophet for Georgia
Completed Georgia
Training Prophet for Illinois
Completed Illinois
Training Prophet for Indiana
Completed Indiana
Training Prophet for Iowa
Completed Iowa
Training Prophet for Kansas
Completed Kansas
Training Prophet for Kentucky
Completed Kentucky
Training Prophet for Louisiana
Completed Louisiana
Training Prophet for Maine
Completed Maine
Training Prophet for Maryland
Completed Maryland
Training Prophet for Massachusetts
Completed Massachusetts
Training Prophet for Michigan
Completed Michigan
Training Prophet for Minnesota
Completed Minnesota
Training Prophet for Mississippi
Completed Mississippi
Training Prophet f

In [79]:
prophet_results[:5]

[{'State': 'Alabama',
  'Model': 'Prophet',
  'MAE': 5863418.074943701,
  'RMSE': np.float64(6930770.427437876)},
 {'State': 'Arizona',
  'Model': 'Prophet',
  'MAE': 9135175.689613627,
  'RMSE': np.float64(10435869.435545301)},
 {'State': 'Arkansas',
  'Model': 'Prophet',
  'MAE': 2998921.7405056027,
  'RMSE': np.float64(3606141.7802643073)},
 {'State': 'California',
  'Model': 'Prophet',
  'MAE': 24422584.62906648,
  'RMSE': np.float64(29176975.99854204)},
 {'State': 'Colorado',
  'Model': 'Prophet',
  'MAE': 6466984.664814299,
  'RMSE': np.float64(7738475.665527627)}]

In [80]:
prophet_results_df = pd.DataFrame(
    prophet_results
)

prophet_results_df.head()

,State,Model,MAE,RMSE
0,Alabama,Prophet,5.863418e+06,6.930770e+06
1,Arizona,Prophet,9.135176e+06,1.043587e+07
2,Arkansas,Prophet,2.998922e+06,3.606142e+06
3,California,Prophet,2.442258e+07,2.917698e+07
4,Colorado,Prophet,6.466985e+06,7.738476e+06


In [81]:
features = [

    'lag_1',

    'lag_7',

    'lag_30',

    'rolling_mean_7',

    'rolling_std_7',

    'month',

    'week',

    'year',

    'day_of_week',

    'quarter',

    'holiday_flag'

]

In [82]:
xgb_results = []

xgb_models = {}

In [83]:
for state in df['State'].unique():

    print(f"Training XGBoost for {state}")

    # Filter one state
    temp = df[
        df['State'] == state
    ].copy()

    # Train-test split
    train_size = int(
        len(temp) * 0.8
    )

    train = temp.iloc[:train_size]

    test = temp.iloc[train_size:]

    # Features
    X_train = train[features]

    y_train = train['Total']

    X_test = test[features]

    y_test = test['Total']

    try:

        # Create model
        model = XGBRegressor(

            n_estimators=200,

            learning_rate=0.05,

            max_depth=5,

            subsample=0.8,

            colsample_bytree=0.8,

            random_state=42

        )

        # Train
        model.fit(
            X_train,
            y_train
        )

        # Predict
        preds = model.predict(
            X_test
        )

        # Evaluate
        mae, rmse = evaluate_model(

            y_test,

            preds

        )

        # Store results
        xgb_results.append({

            'State': state,

            'Model': 'XGBoost',

            'MAE': mae,

            'RMSE': rmse

        })

        # Save model
        xgb_models[state] = model

        print(f"Completed {state}")

    except Exception as e:

        print(
            f"Error in {state}: {e}"
        )

Training XGBoost for Alabama
Completed Alabama
Training XGBoost for Arizona
Completed Arizona
Training XGBoost for Arkansas
Completed Arkansas
Training XGBoost for California
Completed California
Training XGBoost for Colorado
Completed Colorado
Training XGBoost for Connecticut
Completed Connecticut
Training XGBoost for Florida
Completed Florida
Training XGBoost for Georgia
Completed Georgia
Training XGBoost for Illinois
Completed Illinois
Training XGBoost for Indiana
Completed Indiana
Training XGBoost for Iowa
Completed Iowa
Training XGBoost for Kansas
Completed Kansas
Training XGBoost for Kentucky
Completed Kentucky
Training XGBoost for Louisiana
Completed Louisiana
Training XGBoost for Maine
Completed Maine
Training XGBoost for Maryland
Completed Maryland
Training XGBoost for Massachusetts
Completed Massachusetts
Training XGBoost for Michigan
Completed Michigan
Training XGBoost for Minnesota
Completed Minnesota
Training XGBoost for Mississippi
Completed Mississippi
Training XGBoost f

In [84]:
xgb_results[:5]

[{'State': 'Alabama',
  'Model': 'XGBoost',
  'MAE': 4904829.113674621,
  'RMSE': np.float64(6616437.337343986)},
 {'State': 'Arizona',
  'Model': 'XGBoost',
  'MAE': 16918421.789130438,
  'RMSE': np.float64(19290479.4230972)},
 {'State': 'Arkansas',
  'Model': 'XGBoost',
  'MAE': 2992127.241725104,
  'RMSE': np.float64(3778926.606564146)},
 {'State': 'California',
  'Model': 'XGBoost',
  'MAE': 18346223.252173897,
  'RMSE': np.float64(23847770.80201747)},
 {'State': 'Colorado',
  'Model': 'XGBoost',
  'MAE': 2993453.72194951,
  'RMSE': np.float64(3814010.39887807)}]

In [85]:
xgb_results_df = pd.DataFrame(
    xgb_results
)

xgb_results_df.head()

,State,Model,MAE,RMSE
0,Alabama,XGBoost,4.904829e+06,6.616437e+06
1,Arizona,XGBoost,1.691842e+07,1.929048e+07
2,Arkansas,XGBoost,2.992127e+06,3.778927e+06
3,California,XGBoost,1.834622e+07,2.384777e+07
4,Colorado,XGBoost,2.993454e+06,3.814010e+06


In [86]:
lstm_results = []

lstm_models = {}

lstm_scalers = {}

In [87]:
for state in df['State'].unique():

    print(f"Training LSTM for {state}")

    # Filter one state
    temp = df[
        df['State'] == state
    ].copy()

    # Use only target variable
    values = temp[
        'Total'
    ].values.reshape(-1, 1)

    # Scaling
    scaler = MinMaxScaler()

    scaled_values = scaler.fit_transform(
        values
    )

    # Sequence length
    sequence_length = 8

    X = []

    y = []

    # Create sequences
    for i in range(
        sequence_length,
        len(scaled_values)
    ):

        X.append(
            scaled_values[
                i-sequence_length:i
            ]
        )

        y.append(
            scaled_values[i]
        )

    X = np.array(X)

    y = np.array(y)

    # Train-test split
    train_size = int(
        len(X) * 0.8
    )

    X_train = X[:train_size]

    X_test = X[train_size:]

    y_train = y[:train_size]

    y_test = y[train_size:]

    try:

        # Build model
        model = Sequential()

        model.add(

            LSTM(

                64,

                return_sequences=True,

                input_shape=(
                    X_train.shape[1],
                    1
                )

            )

        )

        model.add(
            Dropout(0.2)
        )

        model.add(
            LSTM(32)
        )

        model.add(
            Dropout(0.2)
        )

        model.add(
            Dense(1)
        )

        # Compile
        model.compile(

            optimizer='adam',

            loss='mse'

        )

        # Train
        model.fit(

            X_train,

            y_train,

            epochs=20,

            batch_size=16,

            verbose=0

        )

        # Predict
        preds = model.predict(
            X_test
        )

        # Inverse scaling
        preds = scaler.inverse_transform(
            preds
        )

        y_test_actual = scaler.inverse_transform(
            y_test
        )

        # Evaluate
        mae, rmse = evaluate_model(

            y_test_actual,

            preds

        )

        # Store results
        lstm_results.append({

            'State': state,

            'Model': 'LSTM',

            'MAE': mae,

            'RMSE': rmse

        })

        # Save model
        lstm_models[state] = model

        # Save scaler
        lstm_scalers[state] = scaler

        print(f"Completed {state}")

    except Exception as e:

        print(
            f"Error in {state}: {e}"
        )

Training LSTM for Alabama
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 360ms/step
Completed Alabama
Training LSTM for Arizona
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 467ms/step
Completed Arizona
Training LSTM for Arkansas
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 300ms/step
Completed Arkansas
Training LSTM for California
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 301ms/step
Completed California
Training LSTM for Colorado
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 300ms/step
Completed Colorado
Training LSTM for Connecticut
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 300ms/step
Completed Connecticut
Training LSTM for Florida
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 328ms/step
Completed Florida
Training LSTM for Georgia
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 326ms/step
Completed Georgia
Training LSTM for Illinois
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 308ms/step
Completed Illinois
Training LSTM for Indiana
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 325ms/step
Completed Indiana
Training LSTM for Iowa
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 318ms/step
Completed Iowa
Training LSTM for Kansas
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 445ms/step
Completed

In [88]:
lstm_results[:5]

[{'State': 'Alabama',
  'Model': 'LSTM',
  'MAE': 7991547.207575769,
  'RMSE': np.float64(8953753.694405925)},
 {'State': 'Arizona',
  'Model': 'LSTM',
  'MAE': 7826820.215909099,
  'RMSE': np.float64(9442123.602319852)},
 {'State': 'Arkansas',
  'Model': 'LSTM',
  'MAE': 3864492.184848482,
  'RMSE': np.float64(4353948.964657544)},
 {'State': 'California',
  'Model': 'LSTM',
  'MAE': 26651447.897727262,
  'RMSE': np.float64(29965641.210559238)},
 {'State': 'Colorado',
  'Model': 'LSTM',
  'MAE': 5514337.093181815,
  'RMSE': np.float64(6131849.151369794)}]

In [89]:
lstm_results_df = pd.DataFrame(
    lstm_results
)

lstm_results_df.head()

,State,Model,MAE,RMSE
0,Alabama,LSTM,7.991547e+06,8.953754e+06
1,Arizona,LSTM,7.826820e+06,9.442124e+06
2,Arkansas,LSTM,3.864492e+06,4.353949e+06
3,California,LSTM,2.665145e+07,2.996564e+07
4,Colorado,LSTM,5.514337e+06,6.131849e+06


In [90]:
all_results = pd.concat([

    sarima_results_df,

    prophet_results_df,

    xgb_results_df,

    lstm_results_df

], ignore_index=True)

In [91]:
all_results.head(20)

,State,Model,MAE,RMSE
0,Alabama,SARIMA,3.345134e+07,3.697218e+07
1,Arizona,SARIMA,3.295845e+07,3.599595e+07
2,Arkansas,SARIMA,1.415968e+07,1.572878e+07
3,California,SARIMA,1.445068e+08,1.642945e+08
4,Colorado,SARIMA,1.927795e+07,2.185173e+07
5,Connecticut,SARIMA,8.529113e+06,9.621937e+06
6,Florida,SARIMA,8.387251e+07,9.277292e+07
7,Georgia,SARIMA,4.273416e+07,4.643447e+07
8,Illinois,SARIMA,3.854438e+07,4.364036e+07
9,Indiana,SARIMA,2.508628e+07,2.762984e+07


In [92]:
all_results.to_csv(

    'all_model_results.csv',

    index=False

)

In [93]:
best_models = all_results.loc[

    all_results.groupby(
        'State'
    )['RMSE'].idxmin()

]

In [94]:
best_models.head(20)

,State,Model,MAE,RMSE
86,Alabama,XGBoost,4.904829e+06,6.616437e+06
130,Arizona,LSTM,7.826820e+06,9.442124e+06
45,Arkansas,Prophet,2.998922e+06,3.606142e+06
89,California,XGBoost,1.834622e+07,2.384777e+07
90,Colorado,XGBoost,2.993454e+06,3.814010e+06
134,Connecticut,LSTM,1.353928e+06,2.007689e+06
135,Florida,LSTM,1.438651e+07,2.105428e+07
50,Georgia,Prophet,1.105760e+07,1.270950e+07
137,Illinois,LSTM,9.177610e+06,1.054488e+07
138,Indiana,LSTM,3.626453e+06,5.537981e+06


In [95]:
future_forecasts = []

In [96]:
from datetime import timedelta

for state in best_models['State']:

    print(f"Forecasting for {state}")

    # Get best model
    best_model = best_models[
        best_models['State'] == state
    ]['Model'].values[0]

    # Filter state data
    temp = df[
        df['State'] == state
    ].copy()

    # Last historical date
    last_date = temp['Date'].max()

    try:

        # ==================================================
        # SARIMA FORECAST
        # ==================================================
        if best_model == 'SARIMA':

            model = sarima_models[state]

            preds = model.forecast(
                steps=8
            )

            for i, pred in enumerate(preds):

                future_forecasts.append({

                    'State': state,

                    'Forecast_Date':
                    last_date + timedelta(
                        weeks=i+1
                    ),

                    'Forecast':
                    float(pred),

                    'Best_Model':
                    'SARIMA'

                })

        # ==================================================
        # PROPHET FORECAST
        # ==================================================
        elif best_model == 'Prophet':

            model = prophet_models[state]

            future = model.make_future_dataframe(

                periods=8,

                freq='W'

            )

            forecast = model.predict(
                future
            )

            preds = forecast[
                'yhat'
            ].tail(8).values

            future_dates = forecast[
                'ds'
            ].tail(8).values

            for date, pred in zip(
                future_dates,
                preds
            ):

                future_forecasts.append({

                    'State': state,

                    'Forecast_Date':
                    date,

                    'Forecast':
                    float(pred),

                    'Best_Model':
                    'Prophet'

                })

        # ==================================================
        # XGBOOST FORECAST
        # ==================================================
        elif best_model == 'XGBoost':

            model = xgb_models[state]

            future_data = temp.copy()

            for week in range(8):

                next_date = (
                    last_date +
                    timedelta(
                        weeks=week+1
                    )
                )

                # Create future row
                new_row = {}

                new_row['lag_1'] = (
                    future_data['Total']
                    .iloc[-1]
                )

                new_row['lag_7'] = (
                    future_data['Total']
                    .iloc[-7]
                )

                new_row['lag_30'] = (
                    future_data['Total']
                    .iloc[-30]
                )

                new_row['rolling_mean_7'] = (
                    future_data['Total']
                    .tail(7)
                    .mean()
                )

                new_row['rolling_std_7'] = (
                    future_data['Total']
                    .tail(7)
                    .std()
                )

                new_row['month'] = (
                    next_date.month
                )

                new_row['week'] = (
                    next_date
                    .isocalendar()[1]
                )

                new_row['year'] = (
                    next_date.year
                )

                new_row['day_of_week'] = (
                    next_date.weekday()
                )

                new_row['quarter'] = (
                    (next_date.month - 1)//3
                ) + 1

                new_row['holiday_flag'] = 0

                # Predict
                pred = model.predict(

                    pd.DataFrame([new_row])

                )[0]

                # Store prediction
                future_forecasts.append({

                    'State': state,

                    'Forecast_Date':
                    next_date,

                    'Forecast':
                    float(pred),

                    'Best_Model':
                    'XGBoost'

                })

                # Add prediction back
                future_row = {

                    'Date': next_date,

                    'Total': pred

                }

                future_data = pd.concat([

                    future_data,

                    pd.DataFrame([future_row])

                ], ignore_index=True)

        # ==================================================
        # LSTM FORECAST
        # ==================================================
        else:

            model = lstm_models[state]

            scaler = lstm_scalers[state]

            values = temp[
                'Total'
            ].values.reshape(-1,1)

            scaled = scaler.transform(
                values
            )

            sequence = scaled[-8:]

            future_preds = []

            for week in range(8):

                X_input = sequence.reshape(
                    1, 8, 1
                )

                pred = model.predict(
                    X_input,
                    verbose=0
                )

                future_preds.append(
                    pred[0][0]
                )

                sequence = np.append(
                    sequence[1:],
                    pred
                )

            preds = scaler.inverse_transform(

                np.array(
                    future_preds
                ).reshape(-1,1)

            )

            for i, pred in enumerate(preds):

                future_forecasts.append({

                    'State': state,

                    'Forecast_Date':
                    last_date + timedelta(
                        weeks=i+1
                    ),

                    'Forecast':
                    float(pred[0]),

                    'Best_Model':
                    'LSTM'

                })

        print(f"Completed {state}")

    except Exception as e:

        print(
            f"Error in {state}: {e}"
        )

Forecasting for Alabama
Completed Alabama
Forecasting for Arizona
Completed Arizona
Forecasting for Arkansas
Completed Arkansas
Forecasting for California
Completed California
Forecasting for Colorado
Completed Colorado
Forecasting for Connecticut
Completed Connecticut
Forecasting for Florida
Completed Florida
Forecasting for Georgia
Completed Georgia
Forecasting for Illinois
Completed Illinois
Forecasting for Indiana
Completed Indiana
Forecasting for Iowa
Completed Iowa
Forecasting for Kansas
Completed Kansas
Forecasting for Kentucky
Completed Kentucky
Forecasting for Louisiana
Completed Louisiana
Forecasting for Maine
Completed Maine
Forecasting for Maryland
Completed Maryland
Forecasting for Massachusetts
Completed Massachusetts
Forecasting for Michigan
Completed Michigan
Forecasting for Minnesota
Completed Minnesota
Forecasting for Mississippi
Completed Mississippi
Forecasting for Missouri
Completed Missouri
Forecasting for Nebraska
Completed Nebraska
Forecasting for Nevada
Complet

In [97]:
forecast_df = pd.DataFrame(
    future_forecasts
)

In [98]:
forecast_df.head(20)

,State,Forecast_Date,Forecast,Best_Model
0,Alabama,2023-12-10,2.059771e+08,XGBoost
1,Alabama,2023-12-17,2.013120e+08,XGBoost
2,Alabama,2023-12-24,1.971794e+08,XGBoost
3,Alabama,2023-12-31,1.962260e+08,XGBoost
4,Alabama,2024-01-07,1.922654e+08,XGBoost
5,Alabama,2024-01-14,1.924903e+08,XGBoost
6,Alabama,2024-01-21,1.925668e+08,XGBoost
7,Alabama,2024-01-28,1.887603e+08,XGBoost
8,Arizona,2023-12-10,2.305516e+08,LSTM
9,Arizona,2023-12-17,2.303828e+08,LSTM


In [99]:
forecast_df.to_csv(

    'future_forecasts.csv',

    index=False

)

In [100]:
from google.colab import files

files.download(
    'future_forecasts.csv'
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [101]:
forecast_df.to_csv(
    'future_forecasts.csv',
    index=False
)

In [102]:
!pip install fastapi uvicorn pyngrok nest-asyncio

In [103]:
from fastapi import FastAPI

import uvicorn

import nest_asyncio

from pyngrok import ngrok

In [104]:
app = FastAPI(
    title='Forecasting API',
    version='1.0'
)

In [105]:
forecast_api_df = pd.read_csv(
    'future_forecasts.csv'
)

In [106]:
@app.get('/')

def home():

    return {

        'message':
        'Forecasting API Running Successfully'

    }

In [107]:
@app.get('/forecast/{state}')

def get_forecast(state: str):

    result = forecast_api_df[

        forecast_api_df['State']
        .str.lower()
        ==
        state.lower()

    ]

    return result.to_dict(
        orient='records'
    )

In [108]:
ngrok.set_auth_token(
    '3DO1UxHzxSTFVY8Ea0lGa8UVUa7_3bkoTAzpAmr6MWAQ1Zyyj'
)

In [109]:
nest_asyncio.apply()

public_url = ngrok.connect(8000)

print(public_url)

uvicorn.run(

    app,

    host='0.0.0.0',

    port=8000

)

NgrokTunnel: "https://dad-privacy-unusual.ngrok-free.dev" -> "http://localhost:8000"


RuntimeError: asyncio.run() cannot be called from a running event loop

In [111]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

# Create tunnel
public_url = ngrok.connect(8000)

print("Public URL:", public_url)

# Function to run API
def run():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )

# Start API in background thread
thread = threading.Thread(target=run)

thread.start()

Public URL: NgrokTunnel: "https://dad-privacy-unusual.ngrok-free.dev" -> "http://localhost:8000"


In [112]:
from google.colab import files

files.download('future_forecasts.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [113]:
files.download('best_models.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [114]:
files.download('all_model_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [115]:
requirements = """

pandas
numpy
matplotlib
scikit-learn
statsmodels
prophet
xgboost
tensorflow
fastapi
uvicorn
pyngrok
joblib
openpyxl

"""

with open('requirements.txt', 'w') as f:

    f.write(requirements)

In [116]:
files.download('requirements.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [117]:
app_code = """

from fastapi import FastAPI
import pandas as pd

app = FastAPI()

# Load forecast data
forecast_df = pd.read_csv(
    'future_forecasts.csv'
)

# Home endpoint
@app.get('/')

def home():

    return {

        'message':
        'Forecast API Running Successfully'

    }

# Forecast endpoint
@app.get('/forecast/{state}')

def get_forecast(state: str):

    result = forecast_df[

        forecast_df['State']
        .str.lower()
        ==
        state.lower()

    ]

    return result.to_dict(
        orient='records'
    )

"""

# Create app.py
with open('app.py', 'w') as file:

    file.write(app_code)

print("app.py created successfully")

app.py created successfully


In [118]:
!ls

 all_model_results.csv	'Forecasting Case- Study (1).xlsx'   requirements.txt
 app.py			'Forecasting Case- Study (2).xlsx'   sample_data
 best_models.csv	'Forecasting Case- Study.xlsx'
 final_forecasts.csv	 future_forecasts.csv
